In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

df = pd.read_csv("../data/train.csv")

labels = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]
X = df["comment_text"].fillna("")
Y = df[labels]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=1000,
            solver="liblinear",
            class_weight="balanced"
        )
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print(classification_report(y_test, pred, target_names=labels, zero_division=0))

               precision    recall  f1-score   support

        toxic       0.66      0.84      0.74      3056
 severe_toxic       0.31      0.79      0.44       321
      obscene       0.72      0.87      0.79      1715
       threat       0.31      0.72      0.43        74
       insult       0.60      0.85      0.70      1614
identity_hate       0.29      0.69      0.41       294

    micro avg       0.60      0.84      0.70      7074
    macro avg       0.48      0.79      0.58      7074
 weighted avg       0.63      0.84      0.71      7074
  samples avg       0.06      0.08      0.07      7074



In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

df = pd.read_csv("../data/train.csv")

labels = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]
X = df["comment_text"].fillna("")
Y = df[labels]

X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

model_char = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        analyzer="char_wb",
        ngram_range=(3,5),
        min_df=2,
        max_df=0.9
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=1000,
            solver="liblinear",
            class_weight="balanced"
        )
    ))
])

model_char.fit(X_train, y_train)
pred_char = model_char.predict(X_test)

print(classification_report(y_test, pred_char, target_names=labels, zero_division=0))

               precision    recall  f1-score   support

        toxic       0.64      0.87      0.74      3056
 severe_toxic       0.28      0.84      0.43       321
      obscene       0.71      0.90      0.79      1715
       threat       0.17      0.73      0.28        74
       insult       0.58      0.89      0.70      1614
identity_hate       0.26      0.76      0.39       294

    micro avg       0.57      0.87      0.69      7074
    macro avg       0.44      0.83      0.55      7074
 weighted avg       0.61      0.87      0.71      7074
  samples avg       0.06      0.08      0.07      7074



In [5]:
from sklearn.metrics import f1_score
import pandas as pd

labels = ["toxic","severe_toxic","obscene","threat","insult","identity_hate"]

compare = pd.DataFrame({
    "label": labels,
    "f1_word": f1_score(y_test, pred, average=None),
    "f1_char": f1_score(y_test, pred_char, average=None),
})

compare["diff_char_minus_word"] = compare["f1_char"] - compare["f1_word"]
compare.sort_values("diff_char_minus_word", ascending=False)

,label,f1_word,f1_char,diff_char_minus_word
2,obscene,0.786903,0.791152,0.004250
0,toxic,0.741647,0.741420,-0.000227
4,insult,0.702910,0.702358,-0.000552
1,severe_toxic,0.441739,0.425532,-0.016207
5,identity_hate,0.407631,0.387879,-0.019752
3,threat,0.429150,0.276923,-0.152227


In [6]:
import random
import re
import pandas as pd
from sklearn.metrics import f1_score

random.seed(42)

def add_noise(text):
    if not isinstance(text, str):
        return ""
    t = text.lower()
    t = re.sub(r"http\S+|www\.\S+", " ", t)
    t = re.sub(r"\s+", " ", t).strip()

    t = t.translate(str.maketrans({"a":"4","e":"3","i":"1","o":"0","s":"5","t":"7"}))
    t = re.sub(r"([a-z])\1{1,}", r"\1\1\1", t)
    t = re.sub(r"([a-z])", r"\1 ", t)
    t = re.sub(r"\s+", " ", t).strip()

    if random.random() < 0.5:
        t = t + " !!!"
    return t

X_test_noisy = X_test.apply(add_noise)

pred_word_noisy = model.predict(X_test_noisy)
pred_char_noisy = model_char.predict(X_test_noisy)

f1_word_clean = f1_score(y_test, pred, average="macro")
f1_char_clean = f1_score(y_test, pred_char, average="macro")
f1_word_noisy = f1_score(y_test, pred_word_noisy, average="macro")
f1_char_noisy = f1_score(y_test, pred_char_noisy, average="macro")

pd.DataFrame({
    "model": ["word_tfidf", "char_tfidf"],
    "macro_f1_clean": [f1_word_clean, f1_char_clean],
    "macro_f1_noisy": [f1_word_noisy, f1_char_noisy],
    "drop": [f1_word_clean - f1_word_noisy, f1_char_clean - f1_char_noisy]
})

,model,macro_f1_clean,macro_f1_noisy,drop
0,word_tfidf,0.584997,0.000326,0.584670
1,char_tfidf,0.554211,0.148527,0.405684


In [7]:
summary = pd.DataFrame({
    "model": ["word_tfidf", "char_tfidf"],
    "macro_f1_clean": [0.584997, 0.554211],
    "macro_f1_noisy": [0.000326, 0.148527],
})

summary["relative_retention"] = summary["macro_f1_noisy"] / summary["macro_f1_clean"]
summary

,model,macro_f1_clean,macro_f1_noisy,relative_retention
0,word_tfidf,0.584997,0.000326,0.000557
1,char_tfidf,0.554211,0.148527,0.267997


In [8]:
import re
import pandas as pd

def redact(text):
    t = str(text)
    t = re.sub(r"\s+", " ", t).strip()
    t = t[:220]
    t = re.sub(r"[A-Za-z]", "*", t)
    return t

idx = X_test.index

y_true = y_test["toxic"].values
y_pred = pd.Series(pred[:, 0], index=idx).values

fn_idx = idx[(y_true == 1) & (y_pred == 0)][:10]
fp_idx = idx[(y_true == 0) & (y_pred == 1)][:10]

fn_samples = X_test.loc[fn_idx].apply(redact).reset_index(drop=True)
fp_samples = X_test.loc[fp_idx].apply(redact).reset_index(drop=True)

pd.DataFrame({
    "false_negatives_toxic_redacted": fn_samples,
    "false_positives_toxic_redacted": fp_samples
})

,false_negatives_toxic_redacted,false_positives_toxic_redacted
0,""" **********? ** ** ******. **** **** ** *** *...","****, *** *** *********! **'** ******* *******..."
1,** *** **** *** *** ********. ***** *******.,"********, ****'* **** ***; ***'** *** **** ***..."
2,""" * ****** *** *******? ***. * ****'* ***** **...",* *** *** **** **** * *** ****-****** **** ***...
3,""" *** ** ** *** ***** *** *****? ** * *** ****...","** *** ***, *** ** *** ***** *** ***, ******* ..."
4,""" **, *** * ****** ***'* ********** ***. ***'*...",***** *** ******* **** ***** *** ********* ***...
5,****** **** ******** *** *** **** **** ** ****...,*** ********* ** ***** ***** ** *** *****?????...
6,"****, ******** ** *** **** ****** * ****** ***...",**** *** ** *** **** ** ****** ********** ****...
7,""" ******* ** **** **** *********** ** **** ***...",""" **'* ** *****, ************* ** ******* ****..."
8,**** ***** *** ***** **** * **** ***** *******...,*** ** ** ** *** **** **** **** ** **** *****
9,"**** ********** ** ****** ******** ***, ** ** ...","*******, ****** **** **** **** *********** ** ..."


In [9]:
import pandas as pd

idx = X_test.index
y_true = y_test["toxic"]
y_pred = pd.Series(pred[:, 0], index=idx)

fn_idx = idx[(y_true == 1) & (y_pred == 0)]
fp_idx = idx[(y_true == 0) & (y_pred == 1)]

stats = pd.DataFrame({
    "group": ["false_negatives", "false_positives"],
    "count": [len(fn_idx), len(fp_idx)],
    "avg_len": [X_test.loc[fn_idx].str.len().mean(), X_test.loc[fp_idx].str.len().mean()],
    "median_len": [X_test.loc[fn_idx].str.len().median(), X_test.loc[fp_idx].str.len().median()],
})

stats

,group,count,avg_len,median_len
0,false_negatives,481,405.935551,199.0
1,false_positives,1313,292.022087,143.0
